In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

random_state = 67
np.random.seed(random_state)

In [ ]:
url = ''
df = pd.read_csv(url)

print(f'Data frame has {df.shape[0]} samples, and {df.shape[1]} features')

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
print(f'Number of missing value per column. \n{df.isna().sum()}')

In [ ]:
df.boxplot(figsize=(15,10))
plt.show()

In [ ]:
sns.pairplot(df)
plt.show()

In [ ]:
df['Territorio'].unique().shape

## Preprocessing

In [ ]:
print(f'there are {df.isna().sum().sum()} null values')
df1 = df.copy().dropna()
print(f'there are {df.isna().sum().sum()} null values')
print(f'Data frame has {df.shape[0]} samples, and {df.shape[1]} features')

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
column_to_transform = ''
df[column_to_transform] = le.fit_transform(df[column_to_transform].value)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
one = OneHotEncoder()
column_to_transform = ''
enc_data = one.fit_transform(df[column_to_transform])
enc_df = pd.DataFrame(enc_data.toarray(), columns=list(one.categories_[0]))
df = df.join(enc_df)
df = df.drop(column_to_transform,axis=1)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
categories = []
oe = OneHotEncoder(categories=categories, dtype=int)
column_to_transform = ''
df[column_to_transform] = one.fit_transform(column_to_transform,axis=1)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
mms = MinMaxScaler()
df_processed = pd.DataFrame(mms.fit_transform(df),columns=df.columns)

In [ ]:
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.pipeline import make_pipeline
pipeline = make_pipeline(PowerTransformer(),StandardScaler())
df_processed = pd.DataFrame(pipeline.fit_transform(df),columns=df.columns)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA()
df_tranformed = pca.fit_transform(df)
pca.explained_variance_ratio_
min_variance = 0.9
variance_cumsum = np.cumsum(pca.explained_variance_ratio_.copy())
cutoff_index = np.argmax(variance_cumsum > min_variance)
df = df[:,:cutoff_index+1]

### Train

In [ ]:
X = df.copy()
n_clusters = [*range(2,11)]

In [ ]:
from sklearn.cluster import KMeans
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import silhouette_score

param_km = [{'n_clusters':[*range(2,11)]}]
pg_km = ParameterGrid(param_km)

report_km = pd.DataFrame([],columns=['n_clusters','inertia','silhouette_score'])

for param in pg_km :
    km = KMeans(n_clusters=param['n_clusters'],random_state=random_state)
    y_km = km.fit_transform(X)
    report_km.loc[len(report_km)]=[
        param['n_clusters'],
        km.inertia_,
        silhouette_score(X,y_km)
    ]

In [ ]:
fix,ax = plt.subplots()
ax.plot(n_clusters,report_km['inertia'],color='red')
ax.set_xlabel('n clusters')
ax.set_ylabel('inertia',color='red')

ax2 = ax.twinx()
ax2.plot(n_clusters,report_km['silhouette_score'],color='blue')
ax2.set_ylabel('silhouette_score',color='blue')
ax2.set_ylim(0,1)

plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

param_ac = [{
    'n_clusters':[*range(2,7)],
    'linkage':['ward','complete','single','average']
}]
pg_ac = ParameterGrid(param_ac)
report_ac = pd.DataFrame([],columns=['n_clusters','linkage','silhouette_score'])

for parm in pg_ac:
    ac = AgglomerativeClustering(n_clusters=param['n_clusters'],linkage=param['linkage'])
    y_ac = ac.fit_predict(X)
    report_ac.loc[len(report_ac)]=[
        param['n_clusters'],
        param['linkage'],
        silhouette_score(X,y_ac)
    ]

display(report_ac)

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import silhouette_score

param_dbs = [{
    'eps': [*range(0.001,1,0.05)],
    'min_samples': [*range(2,10)]
}]
pg_dbs = ParameterGrid(param_dbs)
report_dbs = pd.DataFrame([],columns=['n_clusters','eps','min_samples','silouette_score','unclust%'])

for param in pg_dbs:
    dbs = DBSCAN(eps=param['eps'],min_samples=['min_samples'])
    y_dbs = dbs.fit_predict(X)
    y_dbs_clustered = y_dbs[y_dbs != -1, :]
    X_dbs_clustered = X.loc[y_dbs != -1, :]
    n_cluster = len(np.unique(y_dbs_clustered))
    unclust = 1 - y_dbs_clustered.shape[0]/y_dbs.shape[0]
    if n_cluster > 1 and n_cluster < len(X):
        report_dbs.loc[len(report_dbs)]=[
            n_cluster,
            param['eps'],
            param['min_samples'],
            silhouette_score(X_dbs_clustered, y_dbs_clustered),
            unclust*100
        ]

### Display Data

In [ ]:
clust_size_km = np.unique(y_km,return_counts=True)
pd.DataFrame(clust_size_km[1]).plot.pie(y=0,autopct='%1.1f%%')

In [ ]:
X['cluster']=y_km
sns.pairplot(X,hue='cluster')

In [ ]:
result_km[() and
        () and 
        ]